Configuración, panel y utilidades

In [54]:
import cv2 as cv
import numpy as np
import sys, os, unicodedata
from itertools import combinations

RUTA_IMAGEN = 'Imagen2.jpg'
RUTA_VIDEO = 'video.mp4'      # poner 0 para usar la webcam


def tamano_pantalla():
    try:
        import tkinter as tk
        r = tk.Tk(); r.withdraw()
        w, h = r.winfo_screenwidth(), r.winfo_screenheight()
        r.destroy()
        return int(w), int(h)
    except Exception:
        return 1366, 768

PANT_W, PANT_H = tamano_pantalla()
ANCHO_PANEL = 460


imagen_original = cv.imread(RUTA_IMAGEN)
if imagen_original is None:
    sys.exit(f"Error: No se pudo cargar la imagen en '{RUTA_IMAGEN}'.")

max_w = (PANT_W - ANCHO_PANEL - 90) // 2
max_h = PANT_H - 170
factor = min(max_w / imagen_original.shape[1], max_h / imagen_original.shape[0], 1.0)
imagen = (cv.resize(imagen_original, None, fx=factor, fy=factor, interpolation=cv.INTER_AREA)
          if factor < 1.0 else imagen_original.copy())

alto, ancho = imagen.shape[:2]
size = (ancho, alto)

PAL = dict(fondo_a=(46, 18, 76), fondo_b=(20, 8, 38), tarjeta=(66, 34, 108),
           pista=(96, 64, 140), acento=(168, 85, 247), acento2=(217, 70, 239),
           texto=(245, 235, 255), suave=(196, 170, 232), blanco=(255, 255, 255))

def B(c):
    return (int(c[2]), int(c[1]), int(c[0]))

def rect_redondeado(img, p1, p2, r, color):
    (x1, y1), (x2, y2) = p1, p2
    r = max(1, min(r, (x2 - x1) // 2, (y2 - y1) // 2))
    cv.rectangle(img, (x1 + r, y1), (x2 - r, y2), color, -1, cv.LINE_AA)
    cv.rectangle(img, (x1, y1 + r), (x2, y2 - r), color, -1, cv.LINE_AA)
    for cx, cy in ((x1 + r, y1 + r), (x2 - r, y1 + r), (x1 + r, y2 - r), (x2 - r, y2 - r)):
        cv.circle(img, (cx, cy), r, color, -1, cv.LINE_AA)

def texto_img(img, s, org, escala=0.55, color=None):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    color = color or B(PAL['texto'])
    while escala > 0.3 and org[0] + cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, escala, 1)[0][0] > img.shape[1] - 6:
        escala -= 0.02
    cv.putText(img, s, org, cv.FONT_HERSHEY_SIMPLEX, escala, (20, 5, 30), 3, cv.LINE_AA)
    cv.putText(img, s, org, cv.FONT_HERSHEY_SIMPLEX, escala, color, 1, cv.LINE_AA)

def aviso(img, s):
    texto_img(img, s, (10, img.shape[0] - 14), 0.55, (90, 90, 255))

try:
    from PIL import Image, ImageDraw, ImageFont
    import matplotlib
    _DIR = os.path.join(matplotlib.get_data_path(), 'fonts', 'ttf')
    _FUENTES = {'n': os.path.join(_DIR, 'DejaVuSans.ttf'),
                'b': os.path.join(_DIR, 'DejaVuSans-Bold.ttf'),
                'm': os.path.join(_DIR, 'DejaVuSansMono.ttf')}
    ImageFont.truetype(_FUENTES['n'], 12)
    HAY_PIL = True
except Exception:
    HAY_PIL = False

_cache_fuentes = {}
def _fuente(estilo, tam):
    clave = (estilo, int(tam))
    if clave not in _cache_fuentes:
        _cache_fuentes[clave] = ImageFont.truetype(_FUENTES[estilo], int(tam))
    return _cache_fuentes[clave]

def _pintar_textos(img, textos):
    if HAY_PIL:
        pil = Image.fromarray(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        d = ImageDraw.Draw(pil)
        for s, x, y, tam, col, est, anc, maxw in textos:
            tam = int(tam); f = _fuente(est, tam)
            while maxw and f.getlength(s) > maxw and tam > 8:
                tam -= 1; f = _fuente(est, tam)
            d.text((x, y), s, font=f, fill=tuple(col), anchor='rs' if anc == 'r' else 'ls')
        return cv.cvtColor(np.array(pil), cv.COLOR_RGB2BGR)
    for s, x, y, tam, col, est, anc, maxw in textos:
        s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
        esc = tam / 30.0
        while maxw and cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, esc, 1)[0][0] > maxw and esc > 0.3:
            esc -= 0.02
        w = cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, esc, 1)[0][0]
        cv.putText(img, s, (x - w if anc == 'r' else x, y), cv.FONT_HERSHEY_SIMPLEX,
                   esc, B(col), 1, cv.LINE_AA)
    return img


def slider(n, etiqueta, vmin, vmax, ini, fmt=str):
    return dict(n=n, label=etiqueta, min=vmin, max=max(vmax, vmin + 1), ini=ini, fmt=fmt, switch=False)

def interruptor(n, etiqueta, ini=1, fmt=lambda v: 'Sí' if v else 'No'):
    return dict(n=n, label=etiqueta, min=0, max=1, ini=ini, fmt=fmt, switch=True)

px = lambda v: f'{v:+d} px'
grados = lambda v: f'{v:+d}°'


class Panel:
    CAB, FILA, LIN, PAD = 64, 48, 21, 22

    def __init__(self, ventana, cabecera, controles, n_info=4, pie=None):
        self.ventana, self.cabecera, self.ctrl, self.n_info = ventana, cabecera, controles, n_info
        self.pie = pie or 'ESC salir  ·  R reiniciar  ·  clic dcho: restablecer control'
        self.vals = {c['n']: c['ini'] for c in controles}
        self.drag = None
        h0 = self.CAB + self.FILA * len(controles) + 22 + self.LIN * n_info + 56
        self.u = max(0.6, min(1.0, (PANT_H - 110) / h0))
        self.w, self.h = int(ANCHO_PANEL * self.u), int(h0 * self.u)
        t = np.linspace(0, 1, self.h).reshape(-1, 1, 1)
        grad = np.array(B(PAL['fondo_a'])) * (1 - t) + np.array(B(PAL['fondo_b'])) * t
        self.fondo = np.ascontiguousarray(np.broadcast_to(grad, (self.h, self.w, 3))).astype(np.uint8)
        cv.namedWindow(ventana, cv.WINDOW_AUTOSIZE)
        self.activar()

    def activar(self):
        """Hace que este panel reciba los clics de su ventana (varios paneles pueden compartir ventana)."""
        self.drag = None
        cv.setMouseCallback(self.ventana, self._raton)

    def _y(self, i):  return int((self.CAB + self.FILA * i) * self.u)
    def _x0(self):    return int(self.PAD * self.u)
    def _x1(self):    return self.w - int(self.PAD * self.u)

    def valor(self, n):  return self.vals[n]
    def reiniciar(self): self.vals = {c['n']: c['ini'] for c in self.ctrl}

    def _fijar_x(self, i, x):
        c = self.ctrl[i]; x0, x1 = self._x0(), self._x1()
        t = min(max((x - x0) / float(x1 - x0), 0.0), 1.0)
        self.vals[c['n']] = int(round(c['min'] + t * (c['max'] - c['min'])))

    def _raton(self, ev, x, y, flags, _):
        if ev in (cv.EVENT_LBUTTONDOWN, cv.EVENT_RBUTTONDOWN):
            for i, c in enumerate(self.ctrl):
                y0 = self._y(i)
                if y0 <= y < y0 + int(self.FILA * self.u):
                    if ev == cv.EVENT_RBUTTONDOWN:
                        self.vals[c['n']] = c['ini']
                    elif c['switch']:
                        self.vals[c['n']] = 1 - self.vals[c['n']]
                    else:
                        self.drag = i; self._fijar_x(i, x)
                    break
        elif ev == cv.EVENT_MOUSEMOVE and self.drag is not None:
            if flags & cv.EVENT_FLAG_LBUTTON: self._fijar_x(self.drag, x)
            else: self.drag = None
        elif ev == cv.EVENT_LBUTTONUP:
            self.drag = None

    def dibujar(self, lineas_info):
        u, img, textos = self.u, self.fondo.copy(), []
        x0, x1 = self._x0(), self._x1()
        T = lambda s, x, y, tam, col, est='n', anc='l', mw=None: textos.append((s, int(x), int(y), tam * u, col, est, anc, mw))

        T(self.cabecera, x0, 36 * u, 22, PAL['texto'], 'b', mw=x1 - x0)
        cv.line(img, (x0, int(48 * u)), (x0 + int(64 * u), int(48 * u)), B(PAL['acento2']), max(2, int(3 * u)), cv.LINE_AA)

        for i, c in enumerate(self.ctrl):
            y0, v = self._y(i), self.vals[c['n']]
            T(c['label'], x0, y0 + 20 * u, 14.5, PAL['suave'], mw=(x1 - x0) * 0.62)
            if c['switch']:
                pw, ph = int(46 * u), int(22 * u)
                px2, py1 = x1, y0 + int(6 * u)
                rect_redondeado(img, (px2 - pw, py1), (px2, py1 + ph), ph // 2,
                                B(PAL['acento'] if v else PAL['pista']))
                kx = px2 - ph // 2 - 1 if v else px2 - pw + ph // 2 + 1
                cv.circle(img, (kx, py1 + ph // 2), ph // 2 - 3, B(PAL['blanco']), -1, cv.LINE_AA)
                T(c['fmt'](v), px2 - pw - 10 * u, y0 + 20 * u, 14.5, PAL['texto'], 'b', 'r')
            else:
                T(c['fmt'](v), x1, y0 + 20 * u, 14.5, PAL['texto'], 'b', 'r')
                ty, grosor = y0 + int(36 * u), max(4, int(6 * u))
                t = (v - c['min']) / float(c['max'] - c['min'])
                xk = int(x0 + t * (x1 - x0))
                cv.line(img, (x0, ty), (x1, ty), B(PAL['pista']), grosor, cv.LINE_AA)
                for xx in (x0, x1): cv.circle(img, (xx, ty), grosor // 2, B(PAL['pista']), -1, cv.LINE_AA)
                origen = int(x0 + (-c['min'] / float(c['max'] - c['min'])) * (x1 - x0)) if c['min'] < 0 < c['max'] else x0
                cv.line(img, (origen, ty), (xk, ty), B(PAL['acento']), grosor, cv.LINE_AA)
                cv.circle(img, (xk, ty), int(9 * u), B(PAL['acento2']), -1, cv.LINE_AA)
                cv.circle(img, (xk, ty), int(5 * u), B(PAL['blanco']), -1, cv.LINE_AA)

        yi = self._y(len(self.ctrl)) + int(6 * u)
        yf = yi + int((self.LIN * self.n_info + 20) * u)
        rect_redondeado(img, (x0 - int(8 * u), yi), (x1 + int(8 * u), yf), int(12 * u), B(PAL['tarjeta']))
        for k, s in enumerate(lineas_info):
            T(s, x0 + 4 * u, yi + (24 + self.LIN * k) * u, 13, PAL['texto'], 'm', mw=(x1 - x0) - 8 * u)
        T(self.pie, x0, self.h - 16 * u, 11.5, PAL['suave'], mw=x1 - x0)
        return _pintar_textos(img, textos)

def colocar(panel, ventanas, w=None):
    w = w or ancho
    x = 15
    for v in ventanas:
        cv.moveWindow(v, x, 30); x += w + 20
    cv.moveWindow(panel.ventana, max(x, PANT_W - panel.w - 15), 30)

def ventana_cerrada(nombre):
    try:
        return cv.getWindowProperty(nombre, cv.WND_PROP_VISIBLE) < 1
    except cv.error:
        return True

def alguna_cerrada(*nombres):
    return any(ventana_cerrada(n) for n in nombres)

def cerrar_ventanas():
    cv.destroyAllWindows()
    cv.waitKey(1)


def mat_traslacion(tx, ty):
    return np.float32([[1, 0, tx],
                       [0, 1, ty],
                       [0, 0,  1]])

def mat_rotacion(angulo_grados, cx, cy):
    """Rotación alrededor de (cx, cy). Positivo = antihorario."""
    t = np.radians(angulo_grados)
    c, s = np.cos(t), np.sin(t)
    R = np.float32([[ c, s, 0],
                    [-s, c, 0],
                    [ 0, 0, 1]])
    return mat_traslacion(cx, cy) @ R @ mat_traslacion(-cx, -cy)

def mat_escalado(sx, sy, cx, cy):
    """Escalado (sx, sy) respecto al punto (cx, cy), que queda fijo."""
    S = np.float32([[sx,  0, 0],
                    [ 0, sy, 0],
                    [ 0,  0, 1]])
    return mat_traslacion(cx, cy) @ S @ mat_traslacion(-cx, -cy)

def mat_cizalla(shx, shy, cx, cy):
    """Cizallamiento (shx, shy) respecto al punto (cx, cy)."""
    Sh = np.float32([[1, shx, 0],
                     [shy, 1, 0],
                     [0,   0, 1]])
    return mat_traslacion(cx, cy) @ Sh @ mat_traslacion(-cx, -cy)

def mat_reflexion_ejes(horizontal, vertical, w, h):
    """Reflexión respecto a los ejes medianos. El centro es (w-1)/2 para que coincida píxel a píxel con cv.flip."""
    sx = -1 if horizontal else 1
    sy = -1 if vertical else 1
    S = np.float32([[sx,  0, 0],
                    [ 0, sy, 0],
                    [ 0,  0, 1]])
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    return mat_traslacion(cx, cy) @ S @ mat_traslacion(-cx, -cy)

def mat_reflexion_recta(angulo_grados, x0, y0):
    """Reflexión respecto a la recta que pasa por (x0, y0) con dirección (cos θ, sen θ).
    P' = T(p0) · R(θ) · S_y(-1) · R(θ)^t · T(-p0) · P"""
    t = np.radians(angulo_grados)
    c, s = np.cos(t), np.sin(t)
    R     = np.float32([[c, -s, 0], [s,  c, 0], [0, 0, 1]])
    R_inv = np.float32([[c,  s, 0], [-s, c, 0], [0, 0, 1]])
    Ref_x = np.float32([[1,  0, 0], [0, -1, 0], [0, 0, 1]])
    return mat_traslacion(x0, y0) @ R @ Ref_x @ R_inv @ mat_traslacion(-x0, -y0)


# Validación de puntos
def sin_tres_alineados(pts, eps=1.0):
    """False si hay tres puntos (casi) alineados: no definen una transformación válida."""
    p = np.float64(pts)
    for a, b, c in combinations(p, 3):
        if abs((b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])) < eps:
            return False
    return True

def ventana_convexa(pts):
    """Una ventana de proyección válida es un cuadrilátero convexo (no cruzado) sin vértices alineados."""
    return sin_tres_alineados(pts) and cv.isContourConvex(np.float32(pts).reshape(-1, 1, 2))

def homografia(origen, destino):
    """Matriz proyectiva 3x3, o None si los puntos son degenerados (OpenCV no lanza error en ese caso)."""
    if not (sin_tres_alineados(origen) and sin_tres_alineados(destino)):
        return None
    H = cv.getPerspectiveTransform(np.float32(origen), np.float32(destino))
    return H if np.all(np.isfinite(H)) else None


# 1a
def controles_1a(w, h):
    return [
        slider('tx',  'Traslación X',        -w, w, 0, px),
        slider('ty',  'Traslación Y',        -h, h, 0, px),
        slider('ang', 'Ángulo de giro',      -180, 180, 0, grados),
        slider('cgx', 'Centro de giro X',    0, w, w // 2, lambda v: f'{v} px'),
        slider('cgy', 'Centro de giro Y',    0, h, h // 2, lambda v: f'{v} px'),
        slider('sx',  'Factor de escala X',  1, 400, 100, lambda v: f'{v / 100:.2f}×'),
        slider('sy',  'Factor de escala Y',  1, 400, 100, lambda v: f'{v / 100:.2f}×'),
        interruptor('uni', 'Escalado uniforme', 1),
        slider('csx', 'Centro de escalado X', 0, w, w // 2, lambda v: f'{v} px'),
        slider('csy', 'Centro de escalado Y', 0, h, h // 2, lambda v: f'{v} px'),
    ]

def transformar_2d(frame, panel):
    """Orden: 1) escalado  2) rotación  3) traslación.
    M = T(tx,ty) @ [T(c_giro) @ R(θ) @ T(-c_giro)] @ [T(c_esc) @ S(sx,sy) @ T(-c_esc)]"""
    g = panel.valor
    if g('uni') == 1:
        panel.vals['sy'] = panel.vals['sx']
    sx, sy = g('sx') / 100.0, g('sy') / 100.0
    M = (mat_traslacion(g('tx'), g('ty'))
         @ mat_rotacion(g('ang'), g('cgx'), g('cgy'))
         @ mat_escalado(sx, sy, g('csx'), g('csy')))
    h, w = frame.shape[:2]
    return cv.warpAffine(frame, M[:2, :], (w, h)), M, sx, sy

def lineas_1a(g, M, sx, sy):
    return [f"t=({g('tx')},{g('ty')})  θ={g('ang')}°  s=({sx:.2f},{sy:.2f})",
            "M (3x3):",
            *[" ".join(f"{v:9.2f}" for v in fila) for fila in M]]

def dibujar_centros(vista, cg, cs):
    """Marca el centro de giro (cruz) y el de escalado (aspa) sobre la imagen original."""
    for (x, y), col, forma in ((cg, PAL['acento2'], cv.MARKER_CROSS),
                               (cs, PAL['suave'],   cv.MARKER_TILTED_CROSS)):
        x, y = int(x), int(y)
        cv.drawMarker(vista, (x, y), (20, 5, 30), forma, 30, 5)
        cv.drawMarker(vista, (x, y), B(col), forma, 30, 2)
        cv.circle(vista, (x, y), 9, (20, 5, 30), 4, cv.LINE_AA)
        cv.circle(vista, (x, y), 9, B(col), 2, cv.LINE_AA)

def callback_centros(panel, w, h):
    """Clic izquierdo = centro de giro, clic derecho = centro de escalado."""
    def cb(ev, x, y, flags, param):
        if ev == cv.EVENT_LBUTTONDOWN:
            panel.vals['cgx'], panel.vals['cgy'] = max(0, min(w, x)), max(0, min(h, y))
        elif ev == cv.EVENT_RBUTTONDOWN:
            panel.vals['csx'], panel.vals['csy'] = max(0, min(w, x)), max(0, min(h, y))
    return cb

def callback_arrastre(panel, w, h):
    """Arrastrar con el botón izquierdo traslada la imagen en tiempo real."""
    st = {'on': False, 'ini': (0, 0), 'base': (0, 0)}
    def cb(ev, x, y, flags, param):
        if ev == cv.EVENT_LBUTTONDOWN:
            st.update(on=True, ini=(x, y), base=(panel.vals['tx'], panel.vals['ty']))
        elif ev == cv.EVENT_MOUSEMOVE and st['on']:
            if flags & cv.EVENT_FLAG_LBUTTON:          # si se soltó fuera de la ventana, se corta
                panel.vals['tx'] = max(-w, min(w, st['base'][0] + x - st['ini'][0]))
                panel.vals['ty'] = max(-h, min(h, st['base'][1] + y - st['ini'][1]))
            else:
                st['on'] = False
        elif ev == cv.EVENT_LBUTTONUP:
            st['on'] = False
    return cb


# 1b
NOMBRES_V = ['sup. izq.', 'inf. izq.', 'inf. der.', 'sup. der.']
INI_V = [(0.25, 0.20), (0.25, 0.80), (0.75, 0.65), (0.75, 0.35)]

def controles_1b(w, h):
    ctrls = []
    for i, (nom, (fx, fy)) in enumerate(zip(NOMBRES_V, INI_V), start=1):
        ctrls.append(slider(f'x{i}', f'Vértice {i} ({nom}) · X', 0, w, int(fx * w), lambda v: f'{v} px'))
        ctrls.append(slider(f'y{i}', f'Vértice {i} ({nom}) · Y', 0, h, int(fy * h), lambda v: f'{v} px'))
    ctrls.append(interruptor('fondo', 'Mostrar imagen atenuada de fondo', 1))
    return ctrls

def vertices_ventana(g):
    return np.float32([[g(f'x{i}'), g(f'y{i}')] for i in range(1, 5)])

def proyectar_en_ventana(img, pts_destino, fondo_atenuado=True):
    """Proyecta img dentro del cuadrilátero pts_destino (sup-izq, inf-izq, inf-der, sup-der)."""
    h, w = img.shape[:2]
    pts_origen = np.float32([[0, 0], [0, h], [w, h], [w, 0]])
    Tp = cv.getPerspectiveTransform(pts_origen, pts_destino)
    proyectada = cv.warpPerspective(img, Tp, (w, h))
    mascara = cv.warpPerspective(np.full((h, w), 255, np.uint8), Tp, (w, h))
    tinte = np.full_like(img, B(PAL['fondo_b']))
    if fondo_atenuado:
        lienzo = cv.addWeighted((img * 0.35).astype(np.uint8), 0.85, tinte, 0.15, 0)
    else:
        lienzo = tinte
    dentro = mascara > 127
    lienzo[dentro] = proyectada[dentro]
    pts = pts_destino.astype(np.int32)
    cv.polylines(lienzo, [pts.reshape((-1, 1, 2))], True, B(PAL['acento2']), 3, cv.LINE_AA)
    for i, p in enumerate(pts, start=1):
        cv.circle(lienzo, (int(p[0]), int(p[1])), 7, B(PAL['blanco']), -1, cv.LINE_AA)
        cv.circle(lienzo, (int(p[0]), int(p[1])), 7, B(PAL['acento']), 2, cv.LINE_AA)
        texto_img(lienzo, str(i), (int(p[0]) + 10, int(p[1]) - 10), 0.6)
    return lienzo, Tp

def ventana_invalida(ultima, pts, like):
    """Frame a mostrar cuando la ventana no es válida: último frame bueno + contorno rojo + aviso."""
    salida = ultima.copy() if ultima is not None else np.zeros_like(like)
    cv.polylines(salida, [pts.astype(np.int32).reshape(-1, 1, 2)], True, (90, 90, 255), 2, cv.LINE_AA)
    aviso(salida, 'Ventana no valida: vertices cruzados o alineados')
    return salida


# 1c
K_MAX = 1.0
P_MAX = 0.1
fk = lambda v: f'{v / 100 * K_MAX:+.2f}'
fp = lambda v: f'{v / 100 * P_MAX:+.3f}'

def controles_1c(w, h):
    return [
        slider('k1', 'k1 · radial',      -100, 100, 0, fk),
        slider('k2', 'k2 · radial',      -100, 100, 0, fk),
        slider('k3', 'k3 · radial',      -100, 100, 0, fk),
        slider('p1', 'p1 · tangencial',  -100, 100, 0, fp),
        slider('p2', 'p2 · tangencial',  -100, 100, 0, fp),
        slider('f',  'Distancia focal f (fx = fy)', 1, 2 * w, w, lambda v: f'{v} px'),
        slider('cx', 'Centro óptico cx', 0, w, w // 2, lambda v: f'{v} px'),
        slider('cy', 'Centro óptico cy', 0, h, h // 2, lambda v: f'{v} px'),
        interruptor('rejilla', 'Mostrar rejilla', 1),
    ]

def con_rejilla(img, paso=None):
    h, w = img.shape[:2]
    paso = paso or max(16, w // 16)
    out = img.copy()
    for x in range(0, w, paso):
        cv.line(out, (x, 0), (x, h), B(PAL['acento2']), 1, cv.LINE_AA)
    for y in range(0, h, paso):
        cv.line(out, (0, y), (w, y), B(PAL['acento2']), 1, cv.LINE_AA)
    return out

def aplicar_distorsion(img, k1, k2, p1, p2, k3, f, cx, cy):
    dist_coeffs = np.array([k1, k2, p1, p2, k3], dtype=np.float64)
    matriz_camara = np.float64([[f, 0, cx],
                                [0, f, cy],
                                [0, 0,  1]])
    return cv.undistort(img, matriz_camara, dist_coeffs)

def coeficientes_1c(g):
    return (g('k1') / 100 * K_MAX, g('k2') / 100 * K_MAX, g('p1') / 100 * P_MAX,
            g('p2') / 100 * P_MAX, g('k3') / 100 * K_MAX, g('f'), g('cx'), g('cy'))


# fuente de vídeo (con bucle)
class Fuente:
    """Vídeo o webcam. Al terminar un archivo vuelve al principio, así los controles no se cierran."""
    def __init__(self, origen):
        self.es_archivo = isinstance(origen, str)
        self.cap = cv.VideoCapture(origen)
        ok, primero = self.cap.read() if self.cap.isOpened() else (False, None)
        if not ok:
            self.cap.release()
            raise IOError(f"No se pudo abrir {'el vídeo' if self.es_archivo else 'la cámara'} '{origen}'.")
        f = min(max_w / primero.shape[1], max_h / primero.shape[0], 1.0)
        self.size = (int(primero.shape[1] * f), int(primero.shape[0] * f))
        self._pendiente = primero

    def frame(self):
        if self._pendiente is not None:
            fr, self._pendiente = self._pendiente, None
        else:
            ok, fr = self.cap.read()
            if not ok and self.es_archivo:
                self.cap.set(cv.CAP_PROP_POS_FRAMES, 0)
                ok, fr = self.cap.read()
            if not ok:
                return None
        if (fr.shape[1], fr.shape[0]) != self.size:
            fr = cv.resize(fr, self.size, interpolation=cv.INTER_AREA)
        return fr

    def liberar(self):
        self.cap.release()

print(f"Imagen: {ancho}x{alto} (pantalla {PANT_W}x{PANT_H}) | fuente TrueType: {HAY_PIL}")


Imagen: 365x274 (pantalla 1280x720) | fuente TrueType: True


Apartado 1a: traslación, rotación y escalado

In [55]:
# Orden: 1) Escalado  2) Rotación  3) Traslación
#   M = T(tx,ty) @ [T(c_giro) @ R(θ) @ T(-c_giro)] @ [T(c_esc) @ S(sx,sy) @ T(-c_esc)]
# Las matrices (mat_*) y transformar_2d están en la celda de utilidades.

V_ORIG, V_RES = 'Original', 'Resultado 1a'
panel_1a = Panel('Controles 1a', 'Transformaciones 2D', controles_1a(ancho, alto), n_info=5)

for v in (V_ORIG, V_RES):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_1a, [V_ORIG, V_RES])

try:
    while True:
        g = panel_1a.valor
        res, M, sx, sy = transformar_2d(imagen, panel_1a)

        vista = imagen.copy()
        dibujar_centros(vista, (g('cgx'), g('cgy')), (g('csx'), g('csy')))

        cv.imshow(V_ORIG, vista)
        cv.imshow(V_RES, res)
        cv.imshow(panel_1a.ventana, panel_1a.dibujar(lineas_1a(g, M, sx, sy)))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_1a.ventana, V_ORIG, V_RES):
            break
        if tecla in (ord('r'), ord('R')):
            panel_1a.reiniciar()
finally:
    cerrar_ventanas()


Apartado 1b: ventana de proyección

In [56]:
# Se traza un cuadrilátero (ventana) con 4 vértices y se proyecta la imagen dentro de él.
# Origen: las 4 esquinas de la imagen. Destino: los vértices de la ventana (sliders).
# Tp = cv.getPerspectiveTransform(esquinas, vértices)  ->  cv.warpPerspective
V_ORIG_B, V_PROY = 'Imagen original', 'Proyección'
panel_1b = Panel('Controles 1b', 'Ventana de proyección', controles_1b(ancho, alto), n_info=4)

for v in (V_ORIG_B, V_PROY):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_1b, [V_ORIG_B, V_PROY])

ultima, Tp = None, np.eye(3)

try:
    while True:
        pts_destino = vertices_ventana(panel_1b.valor)

        if ventana_convexa(pts_destino):
            ultima, Tp = proyectar_en_ventana(imagen, pts_destino, panel_1b.valor('fondo') == 1)
            salida = ultima
        else:      # vértices cruzados o alineados: no hay ventana válida
            salida = ventana_invalida(ultima, pts_destino, imagen)

        vista = imagen.copy()
        cv.rectangle(vista, (1, 1), (ancho - 2, alto - 2), B(PAL['acento2']), 3)
        cv.imshow(V_ORIG_B, vista)
        cv.imshow(V_PROY, salida)
        cv.imshow(panel_1b.ventana, panel_1b.dibujar([
            "Matriz proyectiva Tp (3x3):",
            *[" ".join(f"{v:10.4f}" for v in fila) for fila in Tp],
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_1b.ventana, V_ORIG_B, V_PROY):
            break
        if tecla in (ord('r'), ord('R')):
            panel_1b.reiniciar()
finally:
    cerrar_ventanas()


Apartado 1c: distorsión de lente

In [57]:
# Modelo [k1, k2, p1, p2, k3] (k = radiales, p = tangenciales), igual que en los ejemplos:
#   k > 0 -> efecto barril | k < 0 -> efecto almohadilla
V_ORIG_C, V_DIST = 'Original', 'Distorsionada'
panel_1c = Panel('Controles 1c', 'Distorsión de lente', controles_1c(ancho, alto), n_info=3)

for v in (V_ORIG_C, V_DIST):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_1c, [V_ORIG_C, V_DIST])

try:
    while True:
        g = panel_1c.valor
        k1, k2, p1, p2, k3, f, cx, cy = coeficientes_1c(g)

        fuente_c = con_rejilla(imagen) if g('rejilla') == 1 else imagen
        distorsionada = aplicar_distorsion(fuente_c, k1, k2, p1, p2, k3, f, cx, cy)

        cv.imshow(V_ORIG_C, fuente_c)
        cv.imshow(V_DIST, distorsionada)
        cv.imshow(panel_1c.ventana, panel_1c.dibujar([
            f"k1={k1:+.2f}   k2={k2:+.2f}   k3={k3:+.2f}",
            f"p1={p1:+.3f}  p2={p2:+.3f}",
            "k>0: barril   |   k<0: almohadilla",
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_1c.ventana, V_ORIG_C, V_DIST):
            break
        if tecla in (ord('r'), ord('R')):
            panel_1c.reiniciar()
finally:
    cerrar_ventanas()


Apartado Optativo: punto de giro, punto de escalado y traslación con el ratón

In [58]:
# Mismo 1a, pero con el ratón:
#   Ventana original  -> clic izquierdo: centro de giro | clic derecho: centro de escalado
#   Ventana resultado -> arrastrar con el botón izquierdo: traslada la imagen en tiempo real
V_ORIG_R = 'Original (clic izq: giro | clic dcho: escala)'
V_RES_R = 'Resultado (arrastrar para mover)'
panel_raton = Panel('Controles Ratón', 'Transformaciones 2D + ratón', controles_1a(ancho, alto), n_info=5)

for v in (V_ORIG_R, V_RES_R):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_raton, [V_ORIG_R, V_RES_R])

cv.setMouseCallback(V_ORIG_R, callback_centros(panel_raton, ancho, alto))
cv.setMouseCallback(V_RES_R, callback_arrastre(panel_raton, ancho, alto))

try:
    while True:
        g = panel_raton.valor
        res, M, sx, sy = transformar_2d(imagen, panel_raton)

        vista = imagen.copy()
        dibujar_centros(vista, (g('cgx'), g('cgy')), (g('csx'), g('csy')))
        texto_img(vista, 'Clic izq: giro | Clic dcho: escala', (10, 25), 0.5)

        cv.imshow(V_ORIG_R, vista)
        cv.imshow(V_RES_R, res)
        cv.imshow(panel_raton.ventana, panel_raton.dibujar(lineas_1a(g, M, sx, sy)))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_raton.ventana, V_ORIG_R, V_RES_R):
            break
        if tecla in (ord('r'), ord('R')):
            panel_raton.reiniciar()
finally:
    cerrar_ventanas()


Apartado Optativo: parte obligatoria (1a, 1b y 1c) sobre vídeo

In [59]:
# Las tres transformaciones sobre vídeo en tiempo real. Teclas 1 / 2 / 3 para cambiar de apartado.
# El vídeo vuelve a empezar al terminar. En el apartado 1a también funcionan el ratón
# (clic izq/dcho en la ventana de entrada: giro/escala; arrastrar en la de salida: traslación).
try:
    fuente = Fuente(RUTA_VIDEO)
except IOError as err:
    sys.exit(str(err))
w_v, h_v = fuente.size

V_VID_O, V_VID_R = 'Vídeo entrada (clic izq: giro | dcho: escala)', 'Vídeo resultado (arrastrar para mover)'
VENT_PANEL = 'Controles vídeo'
PIE = 'ESC salir · R reiniciar · 1/2/3: apartado 1a/1b/1c'

paneles = {
    1: Panel(VENT_PANEL, 'Vídeo · 1a Transformaciones', controles_1a(w_v, h_v), n_info=5, pie=PIE),
    2: Panel(VENT_PANEL, 'Vídeo · 1b Ventana de proyección', controles_1b(w_v, h_v), n_info=4, pie=PIE),
    3: Panel(VENT_PANEL, 'Vídeo · 1c Distorsión de lente', controles_1c(w_v, h_v), n_info=3, pie=PIE),
}
modo = 1
paneles[modo].activar()

for v in (V_VID_O, V_VID_R):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(paneles[1], [V_VID_O, V_VID_R], w_v)
cv.setMouseCallback(V_VID_O, callback_centros(paneles[1], w_v, h_v))
cv.setMouseCallback(V_VID_R, callback_arrastre(paneles[1], w_v, h_v))

ultima_v, Tp_v = None, np.eye(3)

try:
    while True:
        frame = fuente.frame()
        if frame is None:
            print("No se pudo leer más imágenes de la fuente.")
            break

        p = paneles[modo]
        g = p.valor

        if modo == 1:                                   # 1a
            res, M, sx, sy = transformar_2d(frame, p)
            vista = frame.copy()
            dibujar_centros(vista, (g('cgx'), g('cgy')), (g('csx'), g('csy')))
            info = lineas_1a(g, M, sx, sy)
        elif modo == 2:                                 # 1b
            pts = vertices_ventana(g)
            if ventana_convexa(pts):
                res, Tp_v = proyectar_en_ventana(frame, pts, g('fondo') == 1)
                ultima_v = res
            else:
                res = ventana_invalida(ultima_v, pts, frame)
            vista = frame.copy()
            cv.rectangle(vista, (1, 1), (w_v - 2, h_v - 2), B(PAL['acento2']), 3)
            info = ["Matriz proyectiva Tp (3x3):", *[" ".join(f"{v:10.4f}" for v in fila) for fila in Tp_v]]
        else:                                           # 1c
            k1, k2, p1, p2, k3, f, cx, cy = coeficientes_1c(g)
            vista = con_rejilla(frame) if g('rejilla') == 1 else frame
            res = aplicar_distorsion(vista, k1, k2, p1, p2, k3, f, cx, cy)
            info = [f"k1={k1:+.2f}   k2={k2:+.2f}   k3={k3:+.2f}",
                    f"p1={p1:+.3f}  p2={p2:+.3f}",
                    "k>0: barril   |   k<0: almohadilla"]

        cv.imshow(V_VID_O, vista)
        cv.imshow(V_VID_R, res)
        cv.imshow(VENT_PANEL, p.dibujar(info))

        tecla = cv.waitKey(10) & 0xFF
        if tecla == 27 or alguna_cerrada(VENT_PANEL, V_VID_O, V_VID_R):
            break
        if tecla in (ord('r'), ord('R')):
            p.reiniciar()
        elif tecla in (ord('1'), ord('2'), ord('3')):
            modo = int(chr(tecla))
            paneles[modo].activar()
finally:
    fuente.liberar()
    cerrar_ventanas()


Apartado Optativo: transformación afín con 3 pares de puntos

In [60]:
# 3 clics en la imagen original y 3 clics en la imagen destino -> cv.getAffineTransform + cv.warpAffine
# R: reiniciar | Z: deshacer el último punto | ESC: salir
pts_o3, pts_d3, historial = [], [], []

V_AFIN_O, V_AFIN_D = 'Afin: origen (3 clics)', 'Afin: destino (3 clics) y resultado'
for v in (V_AFIN_O, V_AFIN_D):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
cv.moveWindow(V_AFIN_O, 15, 30)
cv.moveWindow(V_AFIN_D, ancho + 35, 30)

def clic_afin_origen(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_o3) < 3:
        pts_o3.append((x, y)); historial.append(pts_o3)

def clic_afin_destino(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_d3) < 3:
        pts_d3.append((x, y)); historial.append(pts_d3)

cv.setMouseCallback(V_AFIN_O, clic_afin_origen)
cv.setMouseCallback(V_AFIN_D, clic_afin_destino)

try:
    while True:
        img_o = imagen.copy()
        img_d = (imagen * 0.25).astype(np.uint8)          # fondo tenue como guía para colocar los puntos

        if len(pts_o3) == 3:
            cv.polylines(img_o, [np.int32(pts_o3).reshape(-1, 1, 2)], True, B(PAL['acento2']), 1, cv.LINE_AA)
        for i, p in enumerate(pts_o3, 1):
            cv.circle(img_o, p, 6, B(PAL['acento2']), -1, cv.LINE_AA)
            texto_img(img_o, f"P{i}", (p[0] + 8, p[1] - 8), 0.5)
        texto_img(img_o, f"Origen: {len(pts_o3)}/3 clics | Z deshacer | R reiniciar", (10, 25), 0.5)

        if len(pts_o3) == 3 and len(pts_d3) == 3:
            if sin_tres_alineados(pts_o3) and sin_tres_alineados(pts_d3):
                M_afin = cv.getAffineTransform(np.float32(pts_o3), np.float32(pts_d3))
                img_d = cv.warpAffine(imagen, M_afin, size)
            else:
                aviso(img_d, 'Puntos alineados: no definen una transformacion afin')

        if len(pts_d3) == 3:
            cv.polylines(img_d, [np.int32(pts_d3).reshape(-1, 1, 2)], True, B(PAL['acento']), 1, cv.LINE_AA)
        for i, p in enumerate(pts_d3, 1):
            cv.circle(img_d, p, 6, B(PAL['acento']), -1, cv.LINE_AA)
            texto_img(img_d, f"P{i}'", (p[0] + 8, p[1] - 8), 0.5)
        texto_img(img_d, f"Destino: {len(pts_d3)}/3 clics | ESC salir", (10, 25), 0.5)

        cv.imshow(V_AFIN_O, img_o)
        cv.imshow(V_AFIN_D, img_d)

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or alguna_cerrada(V_AFIN_O, V_AFIN_D):
            break
        if tecla in (ord('r'), ord('R')):
            pts_o3.clear(); pts_d3.clear(); historial.clear()
        elif tecla in (ord('z'), ord('Z')) and historial:
            historial.pop().pop()
finally:
    cerrar_ventanas()


Apartado Optativo: transformación proyectiva con 4 pares de puntos (homografía)

In [61]:
# 4 clics en la imagen original y 4 clics en la imagen destino -> homografía + cv.warpPerspective
# R: reiniciar | Z: deshacer el último punto | ESC: salir
pts_o4, pts_d4, historial = [], [], []

V_HOM_O, V_HOM_D = 'Proyectiva: origen (4 clics)', 'Proyectiva: destino (4 clics) y resultado'
for v in (V_HOM_O, V_HOM_D):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
cv.moveWindow(V_HOM_O, 15, 30)
cv.moveWindow(V_HOM_D, ancho + 35, 30)

def clic_hom_origen(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_o4) < 4:
        pts_o4.append((x, y)); historial.append(pts_o4)

def clic_hom_destino(ev, x, y, flags, param):
    if ev == cv.EVENT_LBUTTONDOWN and len(pts_d4) < 4:
        pts_d4.append((x, y)); historial.append(pts_d4)

cv.setMouseCallback(V_HOM_O, clic_hom_origen)
cv.setMouseCallback(V_HOM_D, clic_hom_destino)

try:
    while True:
        img_o = imagen.copy()
        img_d = (imagen * 0.25).astype(np.uint8)

        if len(pts_o4) == 4:
            cv.polylines(img_o, [np.int32(pts_o4).reshape(-1, 1, 2)], True, B(PAL['acento2']), 1, cv.LINE_AA)
        for i, p in enumerate(pts_o4, 1):
            cv.circle(img_o, p, 6, B(PAL['acento2']), -1, cv.LINE_AA)
            texto_img(img_o, f"{i}", (p[0] + 8, p[1] - 8), 0.5)
        texto_img(img_o, f"Origen: {len(pts_o4)}/4 clics | Z deshacer | R reiniciar", (10, 25), 0.5)

        if len(pts_o4) == 4 and len(pts_d4) == 4:
            H = homografia(pts_o4, pts_d4)
            if H is not None:
                img_d = cv.warpPerspective(imagen, H, size)
            else:
                aviso(img_d, 'Hay 3 puntos alineados: no define una homografia')

        if len(pts_d4) == 4:
            cv.polylines(img_d, [np.int32(pts_d4).reshape(-1, 1, 2)], True, B(PAL['acento']), 1, cv.LINE_AA)
        for i, p in enumerate(pts_d4, 1):
            cv.circle(img_d, p, 6, B(PAL['acento']), -1, cv.LINE_AA)
            texto_img(img_d, f"{i}'", (p[0] + 8, p[1] - 8), 0.5)
        texto_img(img_d, f"Destino: {len(pts_d4)}/4 clics | ESC salir", (10, 25), 0.5)

        cv.imshow(V_HOM_O, img_o)
        cv.imshow(V_HOM_D, img_d)

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or alguna_cerrada(V_HOM_O, V_HOM_D):
            break
        if tecla in (ord('r'), ord('R')):
            pts_o4.clear(); pts_d4.clear(); historial.clear()
        elif tecla in (ord('z'), ord('Z')) and historial:
            historial.pop().pop()
finally:
    cerrar_ventanas()


Apartado Optativo: imagen especular mediante matrices homogéneas

In [62]:
# Reflexión respecto a los ejes medianos: S = diag(±1, ±1, 1) entre T(c) y T(-c), con c = ((w-1)/2, (h-1)/2)
# (con (w-1)/2 el resultado coincide píxel a píxel con cv.flip).
panel_esp = Panel('Controles Especular', 'Imagen especular', [
    interruptor('h', 'Reflexión horizontal (eje Y)', 1),
    interruptor('v', 'Reflexión vertical (eje X)', 0),
], n_info=5)

V_ESP_O, V_ESP = 'Original', 'Resultado especular'
for v in (V_ESP_O, V_ESP):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_esp, [V_ESP_O, V_ESP])

try:
    while True:
        ref_h = bool(panel_esp.valor('h'))
        ref_v = bool(panel_esp.valor('v'))

        M_esp = mat_reflexion_ejes(ref_h, ref_v, ancho, alto)
        res_esp = cv.warpAffine(imagen, M_esp[:2, :], size)

        modo_txt = ' + '.join(n for n, activo in (('horizontal', ref_h), ('vertical', ref_v)) if activo) or 'ninguna'

        cv.imshow(V_ESP_O, imagen)
        cv.imshow(V_ESP, res_esp)
        cv.imshow(panel_esp.ventana, panel_esp.dibujar([
            f"Reflexión: {modo_txt}",
            "Matriz de reflexión aplicada:",
            *[" ".join(f"{v:8.2f}" for v in fila) for fila in M_esp],
        ]))

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_esp.ventana, V_ESP_O, V_ESP):
            break
        if tecla in (ord('r'), ord('R')):
            panel_esp.reiniciar()
finally:
    cerrar_ventanas()


Apartado Optativo: reflexión sobre una recta arbitraria

In [63]:
# El eje se define con los sliders (inclinación y distancia al centro de la imagen)
# o con 2 clics en la ventana original (3er clic: empezar otra recta).
LIM_D = max(ancho, alto)
panel_rec = Panel('Controles Eje', 'Reflexión sobre recta', [
    slider('ang_eje', 'Inclinación de la recta', -900, 900, 450, lambda v: f'{v / 10:.1f}°'),
    slider('offset',  'Distancia del eje al centro', -LIM_D, LIM_D, 0, px),
    interruptor('sup', 'Superponer imagen original', 0),
], n_info=5)

V_REC_O = 'Original (2 clics definen la recta)'
V_REC_R = 'Reflexión sobre la recta'
for v in (V_REC_O, V_REC_R):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_rec, [V_REC_O, V_REC_R])

puntos_recta = []

def clic_recta(ev, x, y, flags, param):
    if ev != cv.EVENT_LBUTTONDOWN:
        return
    if len(puntos_recta) == 2:
        puntos_recta.clear()
    puntos_recta.append((x, y))
    if len(puntos_recta) == 2:
        (x1, y1), (x2, y2) = puntos_recta
        if (x1, y1) == (x2, y2):
            puntos_recta.pop(); return
        ang = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if ang > 90: ang -= 180                  # una recta se puede describir con ángulos en [-90, 90]
        elif ang < -90: ang += 180
        t = np.radians(ang)
        d = -np.sin(t) * (x1 - ancho / 2) + np.cos(t) * (y1 - alto / 2)    # distancia con signo al centro
        panel_rec.vals['ang_eje'] = int(round(ang * 10))
        panel_rec.vals['offset'] = int(max(-LIM_D, min(LIM_D, round(d))))

cv.setMouseCallback(V_REC_O, clic_recta)

def dibujar_eje(img, x0, y0, c, s):
    longitud = max(ancho, alto) * 2
    p_a = (int(x0 - c * longitud), int(y0 - s * longitud))
    p_b = (int(x0 + c * longitud), int(y0 + s * longitud))
    cv.line(img, p_a, p_b, (20, 5, 30), 4, cv.LINE_AA)
    cv.line(img, p_a, p_b, B(PAL['acento2']), 2, cv.LINE_AA)

try:
    while True:
        ang = panel_rec.valor('ang_eje') / 10.0
        d = panel_rec.valor('offset')
        c, s = np.cos(np.radians(ang)), np.sin(np.radians(ang))

        # Punto de la recta: centro de la imagen desplazado d píxeles en la dirección perpendicular (-s, c)
        x0, y0 = ancho / 2.0 - d * s, alto / 2.0 + d * c
        M_linea = mat_reflexion_recta(ang, x0, y0)
        reflejada = cv.warpAffine(imagen, M_linea[:2, :], size)
        if panel_rec.valor('sup') == 1:
            reflejada = cv.addWeighted(reflejada, 0.5, imagen, 0.5, 0)

        vista = imagen.copy()
        dibujar_eje(vista, x0, y0, c, s)
        dibujar_eje(reflejada, x0, y0, c, s)
        for p in puntos_recta:
            cv.circle(vista, p, 6, B(PAL['blanco']), -1, cv.LINE_AA)
            cv.circle(vista, p, 6, B(PAL['acento']), 2, cv.LINE_AA)

        cv.imshow(V_REC_O, vista)
        cv.imshow(V_REC_R, reflejada)
        cv.imshow(panel_rec.ventana, panel_rec.dibujar([
            f"Eje: ang={ang:.1f}°  dist={d:+d}px",
            "P' = T·R·S_y·R^t·T^-1·P",
            *[" ".join(f"{v:8.2f}" for v in fila) for fila in M_linea],
        ]))

        tecla = cv.waitKey(30) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_rec.ventana, V_REC_O, V_REC_R):
            break
        if tecla in (ord('r'), ord('R')):
            panel_rec.reiniciar(); puntos_recta.clear()
finally:
    cerrar_ventanas()


Otras aportaciones 1: cizallamiento y efecto de la interpolación y del borde

In [64]:
# Transformación afín general (cizallamiento + giro + escala sobre el centro de la imagen) y comparación
# de los modos de interpolación y de relleno de borde de cv.warpAffine.
#   M = R(θ) @ S(s) @ Sh(shx, shy)   (todas respecto al centro)
INTERPOLACIONES = [('Vecino más cercano', cv.INTER_NEAREST), ('Bilineal', cv.INTER_LINEAR), ('Bicúbica', cv.INTER_CUBIC)]
BORDES = [('Constante (negro)', cv.BORDER_CONSTANT), ('Reflejo', cv.BORDER_REFLECT), ('Replicado', cv.BORDER_REPLICATE)]
cx_img, cy_img = ancho / 2.0, alto / 2.0

panel_sh = Panel('Controles Extra', 'Cizallamiento e interpolación', [
    slider('shx', 'Cizallamiento X', -100, 100, 0, lambda v: f'{v / 100:+.2f}'),
    slider('shy', 'Cizallamiento Y', -100, 100, 0, lambda v: f'{v / 100:+.2f}'),
    slider('ang', 'Ángulo de giro', -180, 180, 0, grados),
    slider('esc', 'Escala', 10, 400, 100, lambda v: f'{v / 100:.2f}×'),
    slider('interp', 'Interpolación', 0, 2, 1, lambda v: INTERPOLACIONES[v][0]),
    slider('borde', 'Relleno del borde', 0, 2, 0, lambda v: BORDES[v][0]),
    interruptor('rejilla', 'Mostrar rejilla', 1),
], n_info=6)

V_SH_O, V_SH = 'Original', 'Resultado'
for v in (V_SH_O, V_SH):
    cv.namedWindow(v, cv.WINDOW_AUTOSIZE)
colocar(panel_sh, [V_SH_O, V_SH])

try:
    while True:
        g = panel_sh.valor
        shx, shy, esc = g('shx') / 100.0, g('shy') / 100.0, g('esc') / 100.0
        nombre_i, flag_i = INTERPOLACIONES[g('interp')]
        nombre_b, flag_b = BORDES[g('borde')]

        M = (mat_rotacion(g('ang'), cx_img, cy_img)
             @ mat_escalado(esc, esc, cx_img, cy_img)
             @ mat_cizalla(shx, shy, cx_img, cy_img))

        fuente_s = con_rejilla(imagen) if g('rejilla') == 1 else imagen
        res = cv.warpAffine(fuente_s, M[:2, :], size, flags=flag_i, borderMode=flag_b)

        cv.imshow(V_SH_O, fuente_s)
        cv.imshow(V_SH, res)
        cv.imshow(panel_sh.ventana, panel_sh.dibujar([
            f"sh=({shx:+.2f},{shy:+.2f})  θ={g('ang')}°  s={esc:.2f}",
            f"{nombre_i}  |  {nombre_b}",
            "M (3x3):",
            *[" ".join(f"{v:9.2f}" for v in fila) for fila in M],
        ]))

        tecla = cv.waitKey(20) & 0xFF
        if tecla == 27 or alguna_cerrada(panel_sh.ventana, V_SH_O, V_SH):
            break
        if tecla in (ord('r'), ord('R')):
            panel_sh.reiniciar()
finally:
    cerrar_ventanas()
